# 04. Deep Learning Exploration: Tokenization Spectrum & N-Gram Lookback Ablation (N=1 to N=6)

**Student Name**: Eric Elikplim Sunu  
**Course**: ICS554 Natural Language Processing · Ashesi University  
**Objective**: Investigate how tokenization strategy and n-gram order interact for **Ewe (Èʋegbe)**: how fast test n-grams become unseen as N grows, whether interpolated Kneser-Ney still improves, and which tokenizer models the same text best when scored per word. Also demonstrates the multi-source harmonization pipeline.

---
### Experimental Matrix
1. **Tokenization Strategies**: Character-level, Whitespace, Unicode NFC Word, Ewe Rule Stemmer, and Subword Byte-Pair Encoding (BPE).
2. **Lookback Horizon ($N=1$ to $N=6$)**: Measuring vocabulary size $|V|$, total sequence lengths, zero-count sparsity rate, test perplexity, and qualitative generation.
3. **Multi-Source Dataset Harmonization**: Ingesting up to 4 disparate Ewe data sources, applying Unicode normalization, removing duplicates, and creating clean train/val/test splits.

In [ ]:
import sys
import random
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure repo root is on python path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.ewe_tokenizers import (
    WhitespaceTokenizer,
    UnicodeWordTokenizer,
    CharacterTokenizer,
    EweRuleStemmerTokenizer,
    SimpleBPETokenizer,
)
from src.experiment_runner import run_ngram_experiment
from src.data_pipeline import merge_and_harmonize_datasets

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", palette="muted")

## 1. Load Experimental Ewe Corpus
We load our curated Ewe sentences covering daily greetings, news, and proverbs.

In [ ]:
ewe_corpus = [
    "Woezɔ loo, miawo katã míedi ŋutifafa le dukɔa me",
    "Efoa nyuie mah? Nyee, mefo nyuie, akpe kaka",
    "Kofi yi suku le Keta egbe ŋdi kaba",
    "Ama fle nuɖuɖu vivi le asime le Ho",
    "Míeyi aƒeme kaba elabena tsi le dzadzam le Aflao",
    "Nufiala fia nu nusrɔ̃lawo nyuie le suku me",
    "Mia dogo le etsɔ me ne Mawu lɔ̃",
    "Devi sia nya nu ŋutɔ le eƒe nusɔsrɔ̃ me",
    "Míedi be míawɔ dɔ le ɖekawɔwɔ me le Ghana ha",
    "Ɖo to nyuie ne nàse nya si gblɔm wole na mi",
    "Agbledeŋu nye dɔ vevi aɖe le miaƒe nutoa me",
    "Míele kuku ɖem na mi be miagbɔ kaba",
    "Ŋutsu la kple nyɔnu la woyi agble me le Kpalime",
    "Mía kplɔlawo le dɔ wɔm be dukɔa nade ŋgɔ",
    "Akpe na mi katã ɖe miaƒe kpekpeɖeŋu ta le dɔa me",
    "Nusrɔ̃lawo katã di be yewoawɔ dɔ nyuie le suku"
]

# Tiny illustration only; the real sweep runs on data/processed/ (see scripts/)
train_texts, val_texts, test_texts = ewe_corpus[:12], ewe_corpus[12:14], ewe_corpus[14:]
print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

## 2. Inspecting Tokenization Behaviors Across Ewe Text
Notice how different tokenizers treat Ewe characters (`ɖ`, `ƒ`, `ɣ`, `ŋ`, `ɔ`, `ɛ`, `ʋ`) and tone diacritics.

In [ ]:
sample_phrase = "Woezɔ loo! Nusrɔ̃lawo le suku me. Efoa nyuie mah?"

bpe_tok = SimpleBPETokenizer(num_merges=15)
bpe_tok.train(ewe_corpus)

tokenizers = {
    "Whitespace": WhitespaceTokenizer(),
    "Unicode Word": UnicodeWordTokenizer(),
    "Character": CharacterTokenizer(),
    "Ewe Stemmer": EweRuleStemmerTokenizer(),
    "Byte-Pair Encoding (BPE)": bpe_tok,
}

print(f"Sample Phrase: '{sample_phrase}'\n")
for name, tok in tokenizers.items():
    t_out = tok.tokenize(sample_phrase)
    print(f"{name:25s} -> Tokens ({len(t_out)}): {t_out[:10]}")

## 3. Sweeping N-Gram Orders ($N=1$ to $N=6$) Across Tokenizers
We evaluate how sparsity increases and perplexity evolves as the lookback horizon scales from Unigram ($N=1$) to 6-gram ($N=6$). The results come from `scripts/run_multi_tokenizer_ablation.py` (interpolated Kneser-Ney, words seen once mapped to `<unk>`, best $N$ chosen on validation).

In [ ]:
import json
results_json_path = REPO_ROOT / "reports" / "results_unified_all_tokenizers.json"

if results_json_path.exists():
    print(f"Loading unified-corpus results from {results_json_path.name}...")
    with open(results_json_path) as f:
        data = json.load(f)
    all_results = []
    for tok_name, orders in data["results"].items():
        all_results.extend(orders)
    df_results = pd.DataFrame(all_results)
    print(f"Loaded {len(df_results)} benchmark rows across {df_results['tokenizer'].nunique()} tokenizers.")
else:
    print("Running live sweep on the tiny curated sample (illustration only)...")
    all_results = []
    for name, tok in tokenizers.items():
        all_results.extend(run_ngram_experiment(train_texts, val_texts, test_texts, tok, max_order=6))
    df_results = pd.DataFrame(all_results)

print(df_results[["tokenizer", "order", "vocab_size", "sparsity_pct", "val_perplexity", "perplexity", "per_word_perplexity"]].to_string(index=False))


## 4. Sparsity and Perplexity as N Grows
Left: the share of test n-grams never seen in training rises quickly with $N$. Right: test perplexity **per word**, the scale on which tokenizers can be compared. With Kneser-Ney smoothing a longer context does not make the model worse: when a long context was never seen, its probability mass passes down to the shorter contexts, so the curves flatten instead of turning back up.

In [ ]:
palette = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # reference slots 1-5, validated (light)
markers = ["o", "s", "D", "^", "v"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="#fcfcfb")
panels = [("sparsity_pct", "Test n-grams never seen in training (%)"), ("per_word_perplexity", "Test perplexity per word (log scale, values above 2,000 off the chart)")]
for ax, (col, title) in zip(axes, panels):
    for (tok, g), color, marker in zip(df_results.groupby("tokenizer", sort=False), palette, markers):
        ax.plot(g["order"], g[col], color=color, marker=marker, linewidth=2, markersize=7, label=tok)
    ax.set_title(title, loc="left", fontsize=12, color="#0b0b0b")
    ax.set_xlabel("N-gram order (N)", color="#52514e")
    ax.set_facecolor("#fcfcfb")
    ax.grid(color="#e1e0d9", linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors="#898781")
axes[1].set_yscale("log")
axes[1].set_ylim(150, 2000)  # the region where the tokenizers actually compete
axes[0].legend(frameon=False)
plt.tight_layout()
plt.savefig(REPO_ROOT / "figures" / "ngram_order_ablation.png", dpi=200)
plt.show()


## 5. Text Generation Across Orders (Unigram to 6-Gram)
Seeded samples from the Unicode Word models. Longer contexts copy longer stretches of training text.

In [ ]:
word_results = df_results[df_results["tokenizer"] == "Unicode Word"]
print("=== Generation Evolution in Ewe (Unicode Word Tokenizer) ===\n")
for _, row in word_results.iterrows():
    print(f"[{row['order_name']} (Sparsity: {row['sparsity_pct']}%)]: {row['sample_generation']}")

## 6. Multi-Source Dataset Harmonization Pipeline
The real splits are built by `scripts/build_ewe_datasets.py`. This cell only demonstrates the same pipeline on two tiny made-up sources in a temporary folder, so nothing is written into `data/`.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    (tmp / "source1_conversational.txt").write_text("Woezɔ loo\nEfoa nyuie mah?\nNyee, mefo nyuie, akpe\nMia dogo le etsɔ me\n", encoding="utf-8")
    (tmp / "source2_news.txt").write_text("Kofi yi suku le Keta\nAma fle nuɖuɖu le asime\nMiawo katã míedi ŋutifafa le dukɔa me\nwoezɔ loo\n", encoding="utf-8")
    summary = merge_and_harmonize_datasets(sorted(tmp.glob("*.txt")), output_dir=tmp / "out", min_words=2)
summary